In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report
)
import xgboost as xgb
import joblib
import json
import os

print("All imports successful")

All imports successful


In [2]:
df = pd.read_csv('../data/processed/features_dataset.csv')

print(f"Dataset loaded. Shape: {df.shape}")
print(f"\nLabel distribution:")
print(df['label'].value_counts())

Dataset loaded. Shape: (235795, 19)

Label distribution:
label
1    134850
0    100945
Name: count, dtype: int64


In [3]:
X = df.drop('label', axis=1)
y = df['label']
feature_names = X.columns.tolist()

print(f"X shape (features): {X.shape}")
print(f"y shape (labels): {y.shape}")
print(f"\nFeature names ({len(feature_names)}):")
for name in feature_names:
    print(f"  - {name}")

X shape (features): (235795, 18)
y shape (labels): (235795,)

Feature names (18):
  - url_length
  - domain_length
  - path_length
  - num_dots
  - num_hyphens
  - num_underscores
  - num_slashes
  - num_at_signs
  - num_question_marks
  - num_equals_signs
  - num_digits
  - uses_https
  - uses_ip_address
  - num_subdomains
  - has_www
  - has_suspicious_keyword
  - num_suspicious_keywords
  - domain_entropy


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set: {len(X_train)} URLs")
print(f"Test set: {len(X_test)} URLs")

Training set: 188636 URLs
Test set: 47159 URLs


In [5]:
print("Training Random Forest...")
print("This may take 1-5 minutes...")

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

print("Random Forest training complete!")

Training Random Forest...
This may take 1-5 minutes...
Random Forest training complete!


In [6]:
rf_predictions = rf_model.predict(X_test)

print("=== Random Forest Results ===")
acc  = accuracy_score(y_test, rf_predictions)
prec = precision_score(y_test, rf_predictions)
rec  = recall_score(y_test, rf_predictions)
f1   = f1_score(y_test, rf_predictions)

print(f"Accuracy:  {acc:.4f} ({acc*100:.2f}%)")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")
print()
print(classification_report(y_test, rf_predictions, target_names=['Legitimate', 'Phishing']))

=== Random Forest Results ===
Accuracy:  0.5719 (57.19%)
Precision: 0.5719
Recall:    1.0000
F1 Score:  0.7277

              precision    recall  f1-score   support

  Legitimate       0.00      0.00      0.00     20189
    Phishing       0.57      1.00      0.73     26970

    accuracy                           0.57     47159
   macro avg       0.29      0.50      0.36     47159
weighted avg       0.33      0.57      0.42     47159



d:\HPD\hybrid-phishing-detection\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\HPD\hybrid-phishing-detection\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\HPD\hybrid-phishing-detection\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result

In [7]:
print("Training XGBoost...")
print("This may take 2-10 minutes...")

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

print("\nXGBoost training complete!")

Training XGBoost...
This may take 2-10 minutes...
[0]	validation_0-logloss:0.68277
[50]	validation_0-logloss:0.68277
[100]	validation_0-logloss:0.68277
[150]	validation_0-logloss:0.68277
[199]	validation_0-logloss:0.68277

XGBoost training complete!


In [8]:
xgb_predictions = xgb_model.predict(X_test)

print("=== XGBoost Results ===")
acc  = accuracy_score(y_test, xgb_predictions)
prec = precision_score(y_test, xgb_predictions)
rec  = recall_score(y_test, xgb_predictions)
f1   = f1_score(y_test, xgb_predictions)

print(f"Accuracy:  {acc:.4f} ({acc*100:.2f}%)")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")
print()
print(classification_report(y_test, xgb_predictions, target_names=['Legitimate', 'Phishing']))

=== XGBoost Results ===
Accuracy:  0.5719 (57.19%)
Precision: 0.5719
Recall:    1.0000
F1 Score:  0.7277

              precision    recall  f1-score   support

  Legitimate       0.00      0.00      0.00     20189
    Phishing       0.57      1.00      0.73     26970

    accuracy                           0.57     47159
   macro avg       0.29      0.50      0.36     47159
weighted avg       0.33      0.57      0.42     47159



d:\HPD\hybrid-phishing-detection\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\HPD\hybrid-phishing-detection\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\HPD\hybrid-phishing-detection\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result

In [9]:
results = {
    'Random Forest': {
        'Accuracy':  accuracy_score(y_test, rf_model.predict(X_test)),
        'Precision': precision_score(y_test, rf_model.predict(X_test)),
        'Recall':    recall_score(y_test, rf_model.predict(X_test)),
        'F1 Score':  f1_score(y_test, rf_model.predict(X_test))
    },
    'XGBoost': {
        'Accuracy':  accuracy_score(y_test, xgb_predictions),
        'Precision': precision_score(y_test, xgb_predictions),
        'Recall':    recall_score(y_test, xgb_predictions),
        'F1 Score':  f1_score(y_test, xgb_predictions)
    }
}

comparison_df = pd.DataFrame(results).T
print("=== Model Comparison ===")
print(comparison_df.round(4).to_string())

best = comparison_df['F1 Score'].idxmax()
print(f"\nBest model by F1 Score: {best}")

=== Model Comparison ===
               Accuracy  Precision  Recall  F1 Score
Random Forest    0.5719     0.5719     1.0    0.7277
XGBoost          0.5719     0.5719     1.0    0.7277

Best model by F1 Score: Random Forest


In [10]:
os.makedirs('../models/saved_models', exist_ok=True)

joblib.dump(rf_model, '../models/saved_models/random_forest_model.pkl')
print("Random Forest saved")

joblib.dump(xgb_model, '../models/saved_models/xgboost_model.pkl')
print("XGBoost saved")

with open('../models/saved_models/feature_names.json', 'w') as f:
    json.dump(feature_names, f)
print("Feature names saved")

print("\nAll models saved to models/saved_models/")
print("Note: .pkl files are gitignored — they stay on your laptop only")

Random Forest saved
XGBoost saved
Feature names saved

All models saved to models/saved_models/
Note: .pkl files are gitignored — they stay on your laptop only


In [11]:
comparison_df.to_csv('../docs/report_notes/model_comparison.csv')
print("Comparison table saved to docs/report_notes/model_comparison.csv")

Comparison table saved to docs/report_notes/model_comparison.csv
